# b02·c07 전체 영상에서 저신뢰 원판 후보 연결하기

같은 베이스라인 모델의 후보를 프레임별로 선택한 결과와 직전 움직임으로 연결한 결과를 비교합니다. 영상마다 첫 프레임에서 시작하며 프레임 번호는 0부터 셉니다. **주 평가**는 기존 라벨 범위를 유지합니다. b02 두 번째 영상의 54번 이후는 화면 밖으로 잘리는 원판이 포함되어 **보조 평가**로 따로 표시합니다.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd

from wlcv.config import ANNOTATIONS, RUNS, VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.full_video import detect_videos, compare_videos, summarize_video_scope
from wlcv.evaluation.plate_track import KEY

video_ids = ["b02_sn_f_01", "b02_sn_h_02", "c07_sn_f_01", "c07_sn_f_02"]
weights = RUNS / "plate_baseline/weights/best.pt"
out = RUNS / "plate_baseline/report/b02_c07_full_video"
out.mkdir(parents=True, exist_ok=True)
review = pd.read_csv(ANNOTATIONS / "b02_c07_plate_review.csv")
display(review.groupby(["video_id", "scope", "visibility"]).size().rename("프레임 수"))

## 1. 원본과 검수 박스 확인

예측을 보기 전에 각 영상의 원본에 일정 간격으로 박스를 맞추고, 사이 프레임은 보간한 후 전체를 한 장씩 검수했습니다. 기존 박스 9장은 원래 좌표를 보존했습니다. b02 두 번째 영상의 79–91번은 가까운 원판의 일부만 보여 정답 박스를 비웠습니다. 두 방법 모두 이 구간도 순서대로 처리하며 정탐률과 미탐 길이 계산에서 판독 불가를 구분합니다.

In [ ]:
def draw_box(ax, coords, color, style="-"):
    x1, y1, x2, y2 = coords
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                           edgecolor=color, linestyle=style, linewidth=2))

fig, axes = plt.subplots(1, 4, figsize=(10, 4), layout="constrained")
for ax, video in zip(axes, video_ids):
    row = review[(review.video_id == video) & (review.frame_index == 35)].iloc[0]
    ax.imshow(read_rgb(VIDEOS / f"{video}.mp4", 35))
    draw_box(ax, row[["x1", "y1", "x2", "y2"]], "#ffcc36")
    ax.set_title(f"{video} · 35", fontsize=10)
    ax.axis("off")
plt.show()

## 2. 같은 후보로 시간 연결 끄기와 켜기

원본 전체에서 추론은 한 번만 합니다. 두 방법 모두 프레임 안에서 신뢰도 0.10 이상인 최고 후보를 우선합니다. 시간 연결은 그런 후보가 없고 마지막 고신뢰 선택이 두 프레임 이내일 때만 작동합니다. 직전 박스 너비 절반을 반지름으로 하여 예상 중심에 가장 가까운 후보를 선택합니다. 후보 하한은 0.001, 입력 크기는 640, 장치는 Mac CPU입니다.

In [ ]:
frames, candidates = detect_videos(weights, video_ids)
detail, checks = compare_videos(frames, candidates, review)
summary = summarize_video_scope(detail)
candidates.to_csv(out / "candidates.csv", index=False)
detail.to_csv(out / "per_frame.csv", index=False)
checks.to_csv(out / "distance_checks.csv", index=False)
summary.to_csv(out / "summary.csv", index=False)
display(summary)

## 3. 신뢰도 0.10을 통과한 후보가 없던 프레임

`radius_px`는 허용 반지름, `nearest_px`는 예상 중심에서 가장 가까운 후보까지의 거리입니다. 마지막 고신뢰 탐지와 너무 멀거나 시작점이 없으면 거리를 계산하지 않습니다. 정답과 IoU 0.50 이상인 연결만 복구로 셉니다.

In [ ]:
strong = detail[detail.method == "YOLO"]
tracking = detail[detail.method == "Past link"].copy()
top_score = candidates.groupby(KEY).confidence.max().rename("max_confidence")
missed_keys = strong.loc[strong.source.eq("missing"), KEY]
low = tracking.merge(missed_keys, on=KEY).join(top_score, on=KEY)
low = low.merge(checks[KEY + ["radius_px", "nearest_px", "inside"]], on=KEY, how="left")
low["decision"] = "outside_radius"
low.loc[low.radius_px.isna(), "decision"] = "no_recent_high"
low.loc[low.candidate_count.eq(0), "decision"] = "no_candidate"
low.loc[low.source.eq("linked"), "decision"] = "linked"
low.to_csv(out / "low_frames.csv", index=False)
display(low[KEY + ["scope", "max_confidence", "decision", "radius_px", "nearest_px", "iou", "match"]].round(3))

### 남은 미탐에 정답 위치 후보가 있었나?

선택이 끝난 뒤에만 모든 후보와 검수 박스의 IoU를 계산합니다. 가장 높은 IoU도 0.50 미만이면 이 프레임은 연결 반지름만 바꿔서 복구할 수 없습니다.

In [ ]:
import torch
from ultralytics.utils.metrics import box_iou

missed = strong[strong.match.eq(False).fillna(False)]
candidate_audit = []
for row in missed.itertuples():
    pool = candidates[(candidates.video_id == row.video_id) & (candidates.frame_index == row.frame_index)]
    item = {"video_id": row.video_id, "frame_index": row.frame_index, "candidate_count": len(pool),
            "best_possible_iou": None, "confidence_at_best_iou": None}
    if not pool.empty:
        target = torch.tensor([[row.gt_x1, row.gt_y1, row.gt_x2, row.gt_y2]], dtype=torch.float32)
        boxes = torch.tensor(pool[["x1", "y1", "x2", "y2"]].to_numpy(), dtype=torch.float32)
        overlaps = box_iou(target, boxes)[0]
        best = int(overlaps.argmax())
        item["best_possible_iou"] = overlaps[best].item()
        item["confidence_at_best_iou"] = pool.iloc[best].confidence
    candidate_audit.append(item)
audit = pd.DataFrame(candidate_audit)
audit.to_csv(out / "candidate_audit.csv", index=False)
display(audit.round(4))

## 4. 어느 프레임이 달라졌나

각 점은 해당 프레임에서 가장 높은 후보의 신뢰도입니다. 주황색 음영은 b02 두 번째 영상의 기존 라벨 범위 바깥, 회색 음영은 박스 판독이 어려운 구간입니다. 초록 원은 IoU까지 확인한 복구, 빨간 ×는 고신뢰 후보가 정답 박스와 어긋난 프레임입니다. 원판이 화면에 있는지와 신뢰도는 별도로 확인합니다.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 7), sharex=True, layout="constrained")
for ax, video in zip(axes, video_ids):
    group = strong[strong.video_id == video].join(top_score, on=KEY)
    linked = low[(low.video_id == video) & low.source.eq("linked") & low.match.fillna(False)]
    wrong = group[group.source.eq("high") & group.match.eq(False).fillna(False)]
    if video == "b02_sn_h_02":
        ax.axvspan(53.5, 115.5, color="#fff1da")
        ax.axvspan(78.5, 91.5, color="#dedede")
    ax.plot(group.frame_index, group.max_confidence, ".-", ms=3, lw=1, color="#31578f")
    ax.axhline(.10, color="#914711", ls="--", lw=1)
    ax.scatter(linked.frame_index, linked.max_confidence, s=50, facecolors="none",
               edgecolors="#04784e", linewidths=1.8, zorder=3)
    ax.scatter(wrong.frame_index, wrong.max_confidence, color="#b92234", marker="x", s=55)
    ax.set(ylim=(-.04, 1.06), yticks=[.1, .5, 1], ylabel=video.replace("_sn_", "\n"))
axes[0].set_title("Highest candidate confidence by frame | green ring: recovered, red x: wrong high")
axes[-1].set_xlabel("Frame (starts at 0)")
fig.savefig(out / "confidence.png", dpi=150)
plt.show()

## 5. 복구와 실패를 원본에서 대조

노란 실선은 검수 박스, 청록 점선은 두 방법 중 시간 연결을 켰을 때의 최종 선택입니다. 선택이 비어 있는 경우에만 빨간 점선으로 최고 신뢰도 후보를 추가합니다. 후보가 기준 아래로 떨어진 장면과, 기준을 통과했어도 다른 위치를 고른 장면을 함께 확인합니다.

In [ ]:
video = "b02_sn_h_02"
examples = [35, 93, 115, 97, 98, 101]
fig, axes = plt.subplots(2, 3, figsize=(9, 8), layout="constrained")
for ax, index in zip(axes.flat, examples):
    row = tracking[(tracking.video_id == video) & (tracking.frame_index == index)].iloc[0]
    ax.imshow(read_rgb(VIDEOS / f"{video}.mp4", index))
    draw_box(ax, row[["gt_x1", "gt_y1", "gt_x2", "gt_y2"]], "#ffce36")
    if row.source != "missing":
        draw_box(ax, row[["x1", "y1", "x2", "y2"]], "#00e5ef", "--")
        label = f"{row.source} {row.confidence:.3f} | IoU {row.iou:.2f}"
    else:
        pool = candidates[(candidates.video_id == video) & (candidates.frame_index == index)]
        label = "no candidate"
        if not pool.empty:
            highest = pool.loc[pool.confidence.idxmax()]
            draw_box(ax, highest[["x1", "y1", "x2", "y2"]], "#ff6479", "--")
            label = f"rejected top {highest.confidence:.3f}"
    ax.set_title(f"Frame {index} | {label}", fontsize=10)
    ax.axis("off")
fig.savefig(out / "reviewed_examples.png", dpi=135)
plt.show()

## 6. 코칭에 쓸 동작 중에 원판을 잘못 따라갔나?

동작 중의 잘못된 박스는 속도나 캐치 위치를 바꿀 수 있어 이 구간을 먼저 봅니다. 1차 수동 이벤트에서 `ready`와 `finish` 프레임을 모두 동작에 포함하고, 준비 전과 종료 후를 각각 보조 지표로 봅니다. 이벤트가 일부만 있으면 확인한 범위만 분류합니다. 예를 들어 `ready`와 `drive`까지 표시된 영상은 두 지점 사이가 동작 중임을 알 수 있습니다. 그 뒤는 **구간 미확인**입니다.

각 방법이 최종 선택한 박스 한 개를 기준으로, 정답과 IoU가 0.50 미만인 프레임을 오탐으로 셉니다. 박스가 비어 있으면 미탐입니다. 오탐률의 분모는 해당 구간에서 원판 정답 박스를 판독할 수 있는 프레임 수입니다. 분모가 0이면 ‘미측정’입니다. 기존 예측 파일이 있으면 첫 설정 셀과 아래 셀만 실행해도 됩니다.

In [ ]:
from wlcv.evaluation.full_video import summarize_phase_false_positives

saved = pd.read_csv(out / "per_frame.csv", dtype={"match": "boolean"})
phase_summary = summarize_phase_false_positives(saved, ANNOTATIONS)
phase_summary.to_csv(out / "phase_false_positives.csv", index=False)

table = phase_summary.rename(columns={
    "method": "방법", "phase": "동작 구간", "frames": "전체 프레임",
    "scored": "판독 가능", "fp_frames": "오탐 프레임", "fp_percent": "오탐률 (%)",
})
display(table.style.format({"오탐률 (%)": "{:.1f}"}, na_rep="미측정").hide(axis="index"))